In [1]:
from google.colab import files
import zipfile, os

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(f'/content/{zip_name}', 'r') as z:
    z.extractall('/content/your_folder')
print("✅ Extracted!")

Saving feature-tuning-mixup-main (2).zip to feature-tuning-mixup-main (2).zip
✅ Extracted!


In [2]:
%%writefile /content/your_folder/feature-tuning-mixup-main/feature-tuning-mixup-main/feature-tuning-mixup-main/ftm_attack.py

import torch
import torch.nn as nn
import torch.nn.functional as F


class FeatureTuningTabular(nn.Module):
    def __init__(self, model, mix_prob=0.5, ftm_beta=0.1, device="cpu"):
        super().__init__()
        self.model = model
        self.device = device
        self.mix_prob = mix_prob
        self.ftm_beta = ftm_beta
        self.record = False
        self.outputs = {}
        self.outputs_tuning = {}
        self.forward_hooks = []
        linears = [m for m in model.modules() if isinstance(m, nn.Linear)]
        for idx, m in enumerate(linears):
            self.forward_hooks.append(m.register_forward_hook(self._hook_fn(idx)))

    def _hook_fn(self, layer_idx):
        def hook(module, inp, out):
            if self.record:
                self.outputs[layer_idx] = out.detach()
                self.outputs_tuning[layer_idx] = torch.randn_like(out).to(self.device)
                return out
            if layer_idx not in self.outputs_tuning or torch.rand(1).item() > self.mix_prob:
                return out
            feat_norm   = out.norm(p=2, dim=1, keepdim=True) + 1e-8
            tuning      = self.outputs_tuning[layer_idx]
            tuning_norm = tuning.norm(p=2, dim=1, keepdim=True) + 1e-8
            perturbed   = out + (tuning * (self.ftm_beta * feat_norm / tuning_norm))
            a = torch.rand(out.size(0), 1, device=self.device) * 0.5
            return (1 - a) * perturbed + a * self.outputs[layer_idx]
        return hook

    def remove_hooks(self):
        for h in self.forward_hooks:
            h.remove()
        self.forward_hooks = []

    def forward(self, x):
        return self.model(x)


class CWLoss(nn.Module):
    def __init__(self, target, kappa=0.0):
        super().__init__()
        self.target = target
        self.kappa  = kappa

    def forward(self, logits):
        B = logits.size(0)
        target_logit = logits.gather(1, self.target.view(-1, 1)).squeeze(1)
        mask = torch.ones_like(logits, dtype=torch.bool)
        mask.scatter_(1, self.target.view(-1, 1), False)
        best_other = logits[mask].view(B, -1).max(dim=1)[0]
        return torch.clamp(best_other - target_logit + self.kappa, min=0).mean()


def variance_tuning_grad(tuned_model, loss_fn, x_adv, base_grad,
                         num_samples=10, radius=0.05):
    grad_acc = torch.zeros_like(x_adv)
    for _ in range(num_samples):
        noise = torch.empty_like(x_adv).uniform_(-radius, radius)
        x_nb  = (x_adv + noise).detach().requires_grad_(True)
        g_nb  = torch.autograd.grad(loss_fn(tuned_model(x_nb)), x_nb,
                                    retain_graph=False, create_graph=False)[0]
        grad_acc += g_nb
    return base_grad + grad_acc / num_samples


def ftm_attack(model, x, target_label, epsilon=1.0, alpha=0.05, iterations=400,
               device="cpu", mu=1.0, use_nesterov=True, use_variance=True,
               vt_samples=10, vt_radius=0.05, loss_type="cw",
               input_diversity=True, dropout_p=0.1):

    model.eval()
    x            = x.to(device)
    target_label = target_label.to(device)
    x_min = x - epsilon
    x_max = x + epsilon

    tuned_model = FeatureTuningTabular(model, device=device).to(device)
    tuned_model.record = True
    with torch.no_grad():
        _ = tuned_model(x)
    tuned_model.record = False

    loss_fn = CWLoss(target_label).to(device) if loss_type == "cw" else \
              (lambda logits: F.cross_entropy(logits, target_label))

    x_adv = x.clone().detach()
    g     = torch.zeros_like(x_adv)

    for t in range(iterations):
        if use_nesterov and t > 0:
            x_nes = torch.clamp(x_adv + mu * alpha * g.sign(), x_min, x_max)
        else:
            x_nes = x_adv.clone()

        x_nes = x_nes.detach().requires_grad_(True)
        x_in  = x_nes * (torch.rand_like(x_nes) > dropout_p).float() \
                if input_diversity else x_nes

        loss   = loss_fn(tuned_model(x_in))
        grad_x = torch.autograd.grad(loss, x_nes,
                                     retain_graph=False, create_graph=False)[0]

        if use_variance:
            grad_x = variance_tuning_grad(tuned_model, loss_fn, x_nes,
                                          grad_x, vt_samples, vt_radius)

        grad_norm = torch.sum(torch.abs(grad_x), dim=1, keepdim=True) + 1e-8
        g         = mu * g + grad_x / grad_norm
        x_adv     = torch.clamp(x_adv - alpha * g.sign(), x_min, x_max).detach()

        if t % 100 == 0:
            with torch.no_grad():
                tuned_model.mix_prob = 0
                acc = (model(x_adv).argmax(dim=1) == target_label).float().mean().item() * 100
                tuned_model.mix_prob = 0.5
                print(f"[FTM] Iter {t:3d} | Loss: {loss.item():.4f} | Succ: {acc:.1f}%")

    tuned_model.remove_hooks()
    return x_adv.detach()

Overwriting /content/your_folder/feature-tuning-mixup-main/feature-tuning-mixup-main/feature-tuning-mixup-main/ftm_attack.py


In [3]:

%%writefile /content/your_folder/feature-tuning-mixup-main/feature-tuning-mixup-main/feature-tuning-mixup-main/models.py

import torch.nn as nn


class MLP_Surrogate(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)


class MLP_Target(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.Tanh(),
            nn.Linear(128, 64),
            nn.Tanh(),
            nn.Linear(64, 32),
            nn.Tanh(),
            nn.Linear(32, 2)
        )

    def forward(self, x):
        return self.net(x)

Overwriting /content/your_folder/feature-tuning-mixup-main/feature-tuning-mixup-main/feature-tuning-mixup-main/models.py


In [4]:
%%writefile /content/your_folder/feature-tuning-mixup-main/feature-tuning-mixup-main/feature-tuning-mixup-main/main.py

import argparse
from datetime import datetime
import torch
import numpy as np
import random

from Data_loader import load_data
from preprocessing import preprocess
from models import MLP_Surrogate, MLP_Target
from train_models import train_mlp, train_sklearn_models
from ftm_attack import ftm_attack
from evaluate import evaluate_attack
from sklearn.model_selection import train_test_split

now = datetime.now()
today_string = now.strftime("%Y-%m-%d|%H:%M:%S")


def main(args):
    print(today_string)
    print(args)
    print("#" * 50)

    seed = args.seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    print("Loading dataset...")
    df = load_data()
    print("Preprocessing...")
    X, y = preprocess(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed)

    input_dim = X.shape[1]

    print("Training models...")
    surrogate  = train_mlp(MLP_Surrogate(input_dim), X_train, y_train)
    target_mlp = train_mlp(MLP_Target(input_dim), X_train, y_train)
    lr, rf     = train_sklearn_models(X_train, y_train)
    print("Models trained.")

    print("Selecting attack samples...")
    X_tensor = torch.tensor(X_test, dtype=torch.float32)
    y_tensor = torch.tensor(y_test.values, dtype=torch.long)

    with torch.no_grad():
        preds = surrogate(X_tensor).argmax(dim=1)

    correct_mask = (preds.numpy() == y_test.to_numpy())
    final_mask   = (y_test.to_numpy() == 0) & correct_mask
    x_attack     = X_tensor[final_mask]
    y_attack     = y_tensor[final_mask]
    y_target     = torch.ones(len(x_attack), dtype=torch.long)

    print(f"Number of attack samples: {len(x_attack)}")
    print("Generating adversarial examples with FTM attack...")

    x_adv = ftm_attack(
        surrogate, x_attack, y_target,
        epsilon=args.epsilon,
        alpha=args.alpha,
        iterations=args.iterations,
        mu=1.0,
        use_nesterov=True,
        use_variance=True,
        vt_samples=10,
        vt_radius=0.05,
        loss_type="cw",
        input_diversity=True,
        dropout_p=0.1,
    )

    print("Evaluating attack results...")
    evaluate_attack(
        {"surrogate": surrogate, "mlp_target": target_mlp,
         "logistic_regression": lr, "random_forest": rf},
        x_clean=x_attack, x_adv=x_adv,
        y_true=y_attack.numpy(), y_target=y_target.numpy()
    )
    print("DONE")


def argument_parsing():
    parser = argparse.ArgumentParser()
    parser.add_argument("--epsilon",    type=float, default=1.0)
    parser.add_argument("--alpha",      type=float, default=0.05)
    parser.add_argument("--iterations", type=int,   default=400)
    parser.add_argument("--seed",       type=int,   default=42)
    return parser


if __name__ == "__main__":
    args = argument_parsing().parse_args()
    main(args)

Overwriting /content/your_folder/feature-tuning-mixup-main/feature-tuning-mixup-main/feature-tuning-mixup-main/main.py


In [5]:
import os
base = '/content/your_folder/feature-tuning-mixup-main/feature-tuning-mixup-main/feature-tuning-mixup-main/'

checks = [
    ('ftm_attack.py', 'CWLoss'),
    ('ftm_attack.py', 'use_nesterov'),
    ('ftm_attack.py', 'x_adv - alpha'),   # sign fix check
    ('main.py',       'use_nesterov=True'),
    ('models.py',     'Tanh'),
]
for fname, keyword in checks:
    with open(base + fname) as f:
        content = f.read()
    print("✅" if keyword in content else "❌ MISSING", f"{fname}: {keyword}")

✅ ftm_attack.py: CWLoss
✅ ftm_attack.py: use_nesterov
✅ ftm_attack.py: x_adv - alpha
✅ main.py: use_nesterov=True
✅ models.py: Tanh


In [6]:
import subprocess
from datetime import datetime
from google.colab import files

base = '/content/your_folder/feature-tuning-mixup-main/feature-tuning-mixup-main/feature-tuning-mixup-main/'

result = subprocess.run(
    ['python', 'main.py',
     '--epsilon',    '1.0',
     '--alpha',      '0.05',
     '--iterations', '400'],
    capture_output=True, text=True, cwd=base
)

# Print to screen
print(result.stdout)
if result.stderr:
    print("ERRORS:\n", result.stderr)

# Save to file
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_path = f'/content/result_{timestamp}.txt'

with open(save_path, 'w') as f:
    f.write(result.stdout)

print(f"\n✅ Saved to: {save_path}")

# Download automatically
files.download(save_path)

2026-04-26|15:28:25
Namespace(epsilon=1.0, alpha=0.05, iterations=400, seed=42)
##################################################
Loading dataset...
Preprocessing...
Training models...
Models trained.
Selecting attack samples...
Number of attack samples: 6745
Generating adversarial examples with FTM attack...
[FTM] Iter   0 | Loss: 0.9762 | Succ: 0.0%
[FTM] Iter 100 | Loss: 0.6128 | Succ: 19.2%
[FTM] Iter 200 | Loss: 0.4118 | Succ: 26.1%
[FTM] Iter 300 | Loss: 0.3036 | Succ: 26.7%
Evaluating attack results...

Model                      Clean Acc    Adv Acc   Atk Succ
surrogate                     100.0%      72.8%      27.2%
mlp_target                     80.3%       0.7%      99.3%
logistic_regression            93.6%       9.7%      90.3%
random_forest                  92.8%      41.4%      58.6%
DONE


✅ Saved to: /content/result_20260426_153058.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>